# TANGLE at a glance

TANGLE builds a fiber network by approximating manufacturing as an
ordered, quasi-static recipe. Python describes the geometry and recipe;
Rust validates and packs them; CubeCL keeps the active world resident on
a CPU or GPU while contact, mechanics, and topology evolve.

```text
Cell + materials + fiber collections
                 ↓
      ordered manufacturing recipe
                 ↓
  CubeCL relaxation + adaptive topology
                 ↓
     RunResult → OVITO / BPM
```

This deliberately small two-ply example introduces the ideas developed
individually in tutorials 02–14. All lengths are in meters;
`tangle.units` provides `um`, `mm`, and `nm` multipliers so that
`7 * um` reads as seven micrometers. Building the objects is cheap; the
guarded execution cell is the only part that launches the solver.

In [ ]:
import inspect
from pathlib import Path

import tangle
from tangle.units import mm, um

# Catch a stale compiled extension before a long recipe reaches a
# name or keyword added by a newer notebook.
run_parameters = inspect.signature(tangle.Recipe.run).parameters
if not hasattr(tangle, "RecipeError") or "debug_ovito_path" not in run_parameters:
    raise RuntimeError(
        "This kernel has an older compiled TANGLE extension loaded. "
        "Run the installation cell in tutorial 00, restart the kernel, "
        "select Python (TANGLE), and then run this notebook from the top."
    )

# Recipe construction is cheap; this switch controls the actual solve
# and all filesystem output.
RUN_OVERVIEW = False
output = Path("output/overview")

## 1. Describe the domain and fibers

The cell is periodic in-plane and bounded through-thickness. Because z
is its only bounded axis, z becomes the cell's *stack axis*: the
direction that layer placement, needling, and compaction act along. A
material supplies capsule diameter and an optional admissible bend
radius. A collection holds placed centerlines, preferred rest
centerlines, and metadata before anything enters the simulation.

Here the top large fiber is placed slightly curved but has a straight
rest centerline: relaxation treats that initial curvature as bending.
Bulk and biased collections can instead be made with TANGLE's seeded
fiber generators.

In [ ]:
# Axis order is x, y, z. periodic="xy" represents a repeating sheet;
# bounded z retains physical top and bottom surfaces.
cell = tangle.Cell([1.0 * mm, 1.0 * mm, 1.5 * mm], periodic="xy")
# Diameter controls contact geometry. min_bend_radius is the hard
# admissible-curvature scale, not the preferred rest shape.
small = tangle.Material("small", diameter=7 * um, min_bend_radius=35 * um)
large = tangle.Material("large", diameter=19 * um, min_bend_radius=75 * um)

# Collections are detached local geometry. formation_layer metadata
# lets later recipe operations address manufacturing plies.
bottom = tangle.FiberCollection("bottom ply")
bottom.add_fiber(
    [[-0.35 * mm, -0.10 * mm, 0.0], [0.35 * mm, -0.10 * mm, 0.0]],
    small,
    tags={"family": "x"},
    formation_layer=0,
)
bottom.add_fiber(
    [[0.12 * mm, -0.35 * mm, 0.0], [0.12 * mm, 0.35 * mm, 0.0]],
    small,
    tags={"family": "y"},
    formation_layer=0,
)

top = tangle.FiberCollection("top ply")
# `placed` is the literal initial geometry. Giving it a straight rest
# shape means the visible waviness initially stores bending strain.
placed = [
    [-0.35 * mm, 0.0, 0.0],
    [0.0, 12 * um, 0.0],
    [0.35 * mm, 0.0, 0.0],
]
straight_rest = [
    [-0.35 * mm, 0.0, 0.0],
    [0.0, 0.0, 0.0],
    [0.35 * mm, 0.0, 0.0],
]
top.add_fiber(
    placed,
    large,
    rest_centerline=straight_rest,
    tags={"family": "needle-eligible"},
    formation_layer=1,
)
top.add_fiber(
    [[-0.08 * mm, -0.35 * mm, 0.0], [-0.08 * mm, 0.35 * mm, 0.0]],
    small,
    tags={"family": "y"},
    formation_layer=1,
)

print(len(bottom), len(top), bottom.layer_ids(), top.layer_ids())
print("stack axis:", cell.stack_axis)

## 2. Write the manufacturing recipe

A recipe is an ordered operation list, not a time integrator. It can
insert dormant collections, hold fibers on temporary kinematic
targets, relax, compact the cell, and capture persistent junction
topology. Explicit relaxation gates make the intended sequence
auditable before the expensive run begins.

Operations that hold fibers on targets (layer placement and needling)
return a `HeldTargets` handle. Used in a `with` block, it releases the
targets when the block ends, so every hold is visibly paired with its
release.

This example settles the bottom ply, inserts and lowers the top ply,
performs one visible needling displacement, compacts through-thickness,
performs a strict final `solve`, and finally records selected contacts
as junctions.

In [ ]:
# The recipe inherits stack_axis=2 (z) from the cell's bounded axis.
recipe = tangle.Recipe(cell)

# Insert and settle the first ply before activating the second.
recipe.insert(bottom, translation=[0.5 * mm, 0.5 * mm, 0.35 * mm])
recipe.relax_until_converged(max_iterations=2_000)

recipe.insert(top, translation=[0.5 * mm, 0.5 * mm, 0.80 * mm])

# A SolvePolicy is the acceptance rule for one stage. Penetration is
# hard during assembly, while curvature is only reported so bending
# cannot block deposition before the final strict pass.
assembly_policy = tangle.SolvePolicy(
    "contact-first assembly",
    target_penetration=0.2 * um,
    max_penetration=0.3 * um,
    target_curvature_ratio=2.0,
    hard_curvature=False,
    max_iterations=4_000,
)
# The with block holds layer 1 just above the stack and releases the
# placement targets when it ends. The overrides soften rest bending
# for this one stage only.
with recipe.place_layer_above(1, gap=5 * um, stiffness=0.5, max_translation=5 * um):
    recipe.solve(assembly_policy, tangle.RelaxationOverrides(bend_stiffness=0.2))

# The needle pulls eligible large-fiber vertices; neighboring fibers
# move only when contact transmits that displacement.
with recipe.needle_layer(
    1,
    footprint=tangle.CircularFootprint([0.5 * mm, 0.5 * mm], diameter=120 * um),
    depth=0.25 * mm,
    min_fiber_diameter=15 * um,
    stiffness=0.75,
    max_translation=5 * um,
    max_translation_over_diameter=0.5,
):
    recipe.settle_targets(tolerance=0.2 * um, max_iterations=3_000)

# volume_fraction() shortens only the stack axis (z) by default; any
# other CompactionSettings field can follow as a keyword.
compaction = tangle.CompactionSettings.volume_fraction(
    0.002, max_steps=20, max_relax_windows=4
)
recipe.compact(compaction)

# Strict final gate: both limits are hard and default to the targets.
# The curvature_cleanup preset temporarily favors pulling bends back
# inside the admissible limit.
final_policy = tangle.SolvePolicy(
    "final",
    target_penetration=0.1 * um,
    target_curvature_ratio=1.02,
    max_iterations=4_000,
)
recipe.solve(final_policy, tangle.RelaxationOverrides.preset("curvature_cleanup"))

# Ordinary contacts remain transient until this explicit late capture.
junctions = tangle.JunctionPolicy(
    "late contact capture",
    "bond",
    max_surface_gap=0.2 * um,
    probability=1.0,
    material_pairs=[("large", "small")],
)
recipe.capture_junctions(junctions)

## 3. Configure one resident solve

Configuration objects follow three naming conventions:

- `*Settings` are run-wide: `RelaxationSettings` and
  `CheckpointSettings` apply to every step of the recipe.
- A `*Policy` is the named rule one step follows: the `SolvePolicy`
  and `JunctionPolicy` above.
- `*Overrides` are temporary deltas for the step they are passed to,
  such as `RelaxationOverrides.preset("curvature_cleanup")`.

Every configuration class takes keyword arguments and has
`replace(**changes)` for a modified copy. The global relaxation
settings control contact resolution, fiber mechanics, batching,
backend selection, and adaptive refinement/coarsening. A checkpoint can
preserve the resident formation state and operation cursor for
continuation.

The CPU backend executes the same CubeCL kernels without a GPU. Change
`backend` to `"wgpu"` for a supported accelerator.

In [ ]:
# These are global defaults; a recipe policy or override may adjust
# selected values for one manufacturing stage.
settings = tangle.RelaxationSettings(
    backend="cpu",
    motion_model="flexible",
    penetration_tolerance=0.1 * um,
    # An excess above one: accept curvature ratios up to 1.05.
    curvature_ratio_tolerance=0.05,
    max_step=2 * um,
    # Manufacturing targets are updated between batches. A short batch
    # keeps this small target-heavy recipe responsive without
    # controlling how many OVITO frames are retained.
    iterations_per_batch=6,
    adaptive_segmentation=tangle.AdaptiveSegmentationSettings.profile("balanced"),
)

# The restart stores the recipe cursor and resident solver state, not
# merely a final list of downloaded centerlines.
checkpoint = tangle.CheckpointSettings(
    "tutorial-overview",
    output / "overview.restart",
    interval_iterations=500,
)

print("Recipe operations:")
for number, operation in enumerate(recipe.operations(), start=1):
    print(f"{number:2d}. {operation}")

## 4. Run, inspect, and export

`Recipe.run()` uploads the packed world, executes the ordered recipe,
and returns a `RunResult` with the final geometry plus convergence,
topology, transfer, event, checkpoint, and junction reports. It does
not modify its inputs: the relaxed geometry is a new `Assembly` at
`result.assembly`. If a step cannot meet its policy, `run()` raises
`tangle.RecipeError`, which names the failing operation and the reason.

OVITO output visualizes the relaxed spherocylinders; BPM output
converts them into a bonded-particle model that downstream solvers
such as DIRT can consume. Passing `debug_ovito_path` with no
`debug_snapshot_interval` records concise recipe keyframes. Set an
integer interval only when detailed relaxation frames are needed.

Change `RUN_OVERVIEW` to `True` only when you want to execute the solve.

In [ ]:
if RUN_OVERVIEW:
    # With no snapshot interval, OVITO records recipe milestones only.
    output.mkdir(parents=True, exist_ok=True)
    debug_dump = output / "overview_debug.dump"
    debug_view = output / "overview_debug_view.py"
    debug_session = output / "overview_debug.ovito"
    try:
        result = recipe.run(
            settings,
            checkpoint=checkpoint,
            debug_ovito_path=debug_dump,
            debug_ovito_view_script_path=debug_view,
            debug_ovito_session_path=debug_session,
            debug_ovito_coloring="curvature_ratio",
        )
    except tangle.RecipeError as error:
        # The error identifies the recipe step, not just the symptom.
        print(f"Step {error.operation_index} ({error.operation}) failed: {error.reason}")
        raise

    summary = {
        "converged": result.converged,
        "iterations": result.iterations,
        "max_penetration_um": result.max_penetration / um,
        "max_curvature_ratio": result.max_curvature_ratio,
        "active_segments": result.active_segments,
        "splits": result.segment_splits,
        "merges": result.segment_merges,
        "junctions": result.junction_count,
    }
    print(summary)
    print(f"Debug OVITO: {debug_dump} ({result.debug_ovito_frames} frames)")
    print(f"OVITO view recipe: {debug_view}")
    print(f"OVITO session target: {debug_session}")

    # run() leaves its inputs unchanged; the relaxed state is a new
    # Assembly that can seed a follow-on Recipe.
    relaxed = result.assembly
    print(relaxed.fiber_count, relaxed.cell.lengths)

    # Keep a separate one-frame file for inspecting only the final state.
    result.write_ovito(
        output / "overview_final.dump",
        view_script_path=output / "overview_final_view.py",
        session_path=output / "overview_final.ovito",
        coloring="curvature_ratio",
    )
    # Exact spherocylinders preserve the final active segmentation.
    result.export_bpm(
        output / "overview.data",
        mode="spherocylinders_exact",
        density=1_800.0,
    )

## Where each idea goes next

| Tutorial | Focus |
| --- | --- |
| 02 | Cells, periodic axes, the stack axis, and units |
| 03 | Materials, placed geometry, and rest geometry |
| 04 | Detached fiber collections and metadata |
| 05 | Seeded fiber populations, orientation, and position models |
| 06 | Insertion, selections, and rigid transforms |
| 07 | Relaxation, contact, mechanics, and CubeCL backends |
| 08 | Adaptive refinement and coarsening |
| 09 | Per-stage solve policies, override presets, and `RecipeError` |
| 10 | Layer movement, held targets, and needling |
| 11 | Dynamic compaction targets, paths, and guards |
| 12 | Explicit junction capture |
| 13 | Checkpoints, continuation, and branching |
| 14 | Results, native analysis, contact and neighbor metrics, OVITO, BPM, and PuMA export |